# Lab 6 — Reading a Language Model's Probabilities

**MA553 · Session 7**

Name:

---

In this lab you use the language model from the session, Qwen2.5-1.5B-Instruct.
You look at its tokens and its next-token probabilities, and then you use it to
label 300 movie-review sentences as positive or negative.

**Part A** looks inside the model and measures its accuracy on the 300
sentences.

**Part B** checks whether the model's confidence can be trusted, corrects it,
and measures what happens when one word of the instruction changes.

Work through the cells in order. Six cells are marked **Write this yourself**:
A2, A3, A4, B1, B2 and B3. Each says which variables to create. Later cells use
those names. The **Answer here** cells are short written answers.

Your numbers can differ from the expected numbers in the last digit. Small
differences come from the way each processor does arithmetic.

# Part A — Inside the model

## Setup

This lab uses the movie-review sentences and the model downloaded by
`get_data.py`. If the folders `data/SST-2` and `model/qwen2.5-1.5b-instruct`
are missing, run `python get_data.py` in this folder first.

Run this. Loading the model takes about 30 seconds.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "model/qwen2.5-1.5b-instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)
model.eval()
print("Model loaded.")

## The sentences

This cell loads the 872 test sentences of SST-2 and draws 300 of them at
random. `sentences` holds their text and `labels` their labels: 1 for
positive, 0 for negative.

Run this.

In [ ]:
rows = open("data/SST-2/dev.tsv").read().splitlines()[1:]
rows = [(r.rsplit("\t", 1)[0].strip(), int(r.rsplit("\t", 1)[1])) for r in rows]
chosen = np.random.default_rng(6).choice(len(rows), 300, replace=False)
sentences = [rows[i][0] for i in chosen]
labels = np.array([rows[i][1] for i in chosen])

print("Sentences:", len(sentences), "  positive:", labels.sum(), "  negative:", (labels == 0).sum())
for s, y in zip(sentences[:3], labels[:3]):
    print(y, s)

## A1 — Tokens

The model reads text as tokens. Each token has an id, which is its row in the
model's table of token vectors.

Run this. Then add a sentence of your own in the last line.

In [ ]:
def show_tokens(text):
    ids = tok(text).input_ids
    print([tok.decode([i]) for i in ids])
    print(ids)
    print()

show_tokens("The film was surprisingly good.")
show_tokens("The cinematography was unflinchingly bleak.")
# show_tokens("your own sentence here")

**Answer here.** Which words of your sentence were split into several tokens?
What do these words have in common?

*Answer here.*

## The scores for the next token

This cell gives the model the text "The capital of France is" and keeps the
scores it computes for the next token. `scores` has one number for each token
of the vocabulary.

Run this.

In [ ]:
ids = tok("The capital of France is", return_tensors="pt").input_ids
with torch.no_grad():
    scores = model(ids).logits[0, -1].float()
print("Number of scores:", len(scores))

## A2 — Next-token probabilities ✏️ Write this yourself

**Goal.** Turn the scores into probabilities with softmax, as on the slides:

probability of a token = e^(its score) ÷ (sum of e^(score) over all tokens).

1. Compute the probabilities from `scores`. Use `torch.exp` and `.sum()`.
   Store them in `probs`.
2. Check that `probs` sums to 1.
3. Find the 5 most likely tokens with `torch.topk(probs, 5)`. Print each token
   with `tok.decode([token_id])`, and its probability.

**Expected.** " Paris" 29.2%, " located" 8.4%, " a" 5.8%, " the" 4.0%,
" in" 3.5%.

In [ ]:
# Your code here.

## A3 — Temperature ✏️ Write this yourself

**Goal.** See how the temperature changes the probabilities. The temperature
divides every score before softmax.

1. Write a function `probs_at(T)` that divides `scores` by `T` and returns the
   softmax of the result. You can use `torch.softmax(scores / T, dim=0)`.
2. For T = 0.5, 1 and 2, print the 3 most likely tokens and their
   probabilities.

**Expected.** The probability of " Paris" is 81.6% at T = 0.5, 29.2% at T = 1,
and 1.6% at T = 2.

In [ ]:
# Your code here.

**Answer here.** At T = 2, " Paris" is still the most likely token, but its
probability is only 1.6%. Where did the rest of the probability go?

*Answer here.*

## Sampling answers

This cell asks the model the same question 5 times at temperature 0.7, then
5 times at temperature 1.5. Each answer is sampled: each next token is drawn at
random with the probabilities at that temperature.

Run this. It takes one to two minutes.

In [ ]:
question = [{"role": "user", "content": "Describe the movie Titanic in one short sentence."}]
chat = tok.apply_chat_template(question, add_generation_prompt=True,
                               return_tensors="pt", return_dict=True)
for T in (0.7, 1.5):
    print(f"Temperature {T}")
    for _ in range(5):
        with torch.no_grad():
            out = model.generate(**chat, do_sample=True, temperature=T, top_k=0, top_p=1.0,
                                 max_new_tokens=30)
        print("  -", tok.decode(out[0, chat["input_ids"].shape[1]:], skip_special_tokens=True))
    print()

**Answer here.** Compare the two sets of answers. Which temperature would you
use if you need the same answer each time you ask?

*Answer here.*

## The model's probability for each label

The function `label_probs(sentence, instruction)` asks the model the
instruction about the sentence. It returns two numbers: the probability that
the first word of the answer is "positive", and the probability that it is
"negative". As on the slides, each number adds up the spellings "positive",
"Positive", " positive" and " Positive" (and the same for "negative").

The cell runs it on the 300 sentences and saves the results in
`scores_sentiment.json`. If the file exists, the cell loads it.

**Running time.** About 2 to 5 minutes on a laptop. Run this once.

In [ ]:
SENTIMENT = ("Is the sentiment of this movie review sentence positive or negative? "
             "Answer with one word: positive or negative.")
first_token = lambda words: sorted({tok.encode(w)[0] for w in words})
POS = first_token(["positive", "Positive", " positive", " Positive"])
NEG = first_token(["negative", "Negative", " negative", " Negative"])

def label_probs(sentence, instruction):
    message = [{"role": "user", "content": f"{instruction}\n\nSentence: {sentence}"}]
    x = tok.apply_chat_template(message, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    with torch.no_grad():
        p = torch.softmax(model(**x).logits[0, -1].float(), dim=0)
    return float(p[POS].sum()), float(p[NEG].sum())

def score_all(instruction, file):
    if Path(file).exists():
        return np.array(json.load(open(file)))
    out = []
    for k, s in enumerate(sentences):
        out.append(label_probs(s, instruction))
        if k % 50 == 0:
            print(k, "sentences done")
    json.dump(out, open(file, "w"))
    return np.array(out)

result = score_all(SENTIMENT, "scores_sentiment.json")
p_pos, p_neg = result[:, 0], result[:, 1]
print("First sentence:", sentences[0])
print(f"  p(positive) = {p_pos[0]:.4f}   p(negative) = {p_neg[0]:.4f}   label = {labels[0]}")

## A4 — Accuracy ✏️ Write this yourself

**Goal.** Measure how many of the 300 sentences the model labels correctly.

1. Print the average of `p_pos + p_neg`. This shows how much of its probability
   the model gives to the two answer words.
2. Compute `p = p_pos / (p_pos + p_neg)`. This is the model's probability that
   the sentence is positive, out of the two answers.
3. The predicted label is 1 when `p > 0.5`, and 0 otherwise. Store it in
   `predicted`.
4. Print the accuracy: the share of sentences where `predicted` equals
   `labels`.

**Expected.** The average of `p_pos + p_neg` is 1.00. The accuracy is 93.0%
(279 of 300 sentences).

In [ ]:
# Your code here.

# Part B — Can the probabilities be trusted?

## B1 — Calibration ✏️ Write this yourself

The model's **confidence** in a sentence is the probability of the label it
chose: `p` when it predicts positive, and `1 - p` when it predicts negative.
The model is **calibrated** if, among the sentences where its confidence is
about 90%, about 90% of the labels are correct.

**Goal.** Build the table of the slides: group the sentences by confidence and
compare each group's average confidence with its accuracy.

1. Compute `confidence = np.maximum(p, 1 - p)`.
2. Use the groups 50% to 60%, 60% to 70%, 70% to 80%, 80% to 90%, and 90% to
   100%. The last group includes 100%.
3. For each group, print the number of sentences, their average confidence and
   their accuracy. Also store these three values in the lists `group_n`,
   `group_conf` and `group_acc`. The plot cell after yours uses them.

**Expected.**

| Confidence | Sentences | Average confidence | Accuracy |
|---|---|---|---|
| 50% to 60% | 4 | 53.8% | 25.0% |
| 60% to 70% | 3 | 62.6% | 33.3% |
| 70% to 80% | 4 | 74.3% | 75.0% |
| 80% to 90% | 7 | 86.8% | 57.1% |
| 90% to 100% | 282 | 99.9% | 95.7% |

In [ ]:
# Your code here.

In [ ]:
plt.plot([0.5, 1], [0.5, 1], "--", color="grey", label="calibrated")
plt.plot(group_conf, group_acc, "o-", label="the model")
for n, c, a in zip(group_n, group_conf, group_acc):
    plt.annotate(str(n), (c, a), textcoords="offset points", xytext=(6, -12))
plt.xlabel("average confidence")
plt.ylabel("accuracy")
plt.title("Numbers: sentences in each group")
plt.legend()
plt.show()

**Answer here.** (1) Is the model overconfident, underconfident, or
calibrated in the 90% to 100% group? (2) Why should you not draw conclusions
from the accuracy of the four other groups?

*Answer here.*

## B2 — Correcting the confidence ✏️ Write this yourself

**Temperature scaling** corrects overconfidence by dividing the model's score
by a temperature, as in A3. Here the score of a sentence is

score = log(p_pos) − log(p_neg),

and the corrected probability of positive is 1 ÷ (1 + e^(−score ÷ T)). With
T = 1 this gives back `p`. A temperature above 1 makes the probabilities less
extreme. It never changes which label wins, so the accuracy stays the same.

To choose T, we use the **log loss**: the average of −log(probability given to
the correct label). It is small when the model gives a high probability to the
correct label, and very large when the model is confident and wrong.

We choose T on the first 150 sentences (the **fitting half**), and measure the
result on the last 150 (the **test half**).

**Goal.**

1. Compute `score` for the 300 sentences.
2. Write a function `log_loss(T, s, y)` that returns the log loss of the scores
   `s` with labels `y` at temperature `T`.
3. Try every T in `np.arange(0.5, 10.01, 0.05)` on the fitting half. Store the
   one with the lowest log loss in `best_T`, and print it.
4. On the test half, for T = 1 and for `best_T`, print the number of sentences
   with confidence of 90% or more, their average confidence and their accuracy.

**Expected.** `best_T` is about 2.9. On the test half:

| Confidence 90% to 100% | Sentences | Average confidence | Accuracy |
|---|---|---|---|
| T = 1 (no correction) | 143 | 99.8% | 96.5% |
| T = best_T | 127 | 96.5% | 97.6% |

In [ ]:
# Your code here.

**Answer here.** Why do we choose T on one half of the sentences and measure
the result on the other half?

*Answer here.*

## B3 — Changing one word of the instruction ✏️ Write this yourself

The cell below asks the model again about the same 300 sentences. The only
change is one word of the instruction: "sentiment" becomes "tone".

Run it first. It takes as long as before, and saves `scores_tone.json`.

In [ ]:
TONE = ("Is the tone of this movie review sentence positive or negative? "
        "Answer with one word: positive or negative.")
result = score_all(TONE, "scores_tone.json")
p_pos_tone, p_neg_tone = result[:, 0], result[:, 1]
print("Done.")

**Goal.** Compare the two instructions on the same sentences.

1. Compute the predicted labels with the "tone" instruction, in
   `predicted_tone`, and print the accuracy.
2. Count the sentences whose predicted label changes between the two
   instructions.
3. Among them, count the sentences that the change **fixed** (wrong before,
   right now) and those it **broke** (right before, wrong now).
4. If the change made no real difference, a changed sentence is equally likely
   to be fixed or broken. Test this with `binomtest(fixed, fixed + broken, 0.5)`
   and print its p-value, as on the slides.

**Expected.** Accuracy 92.7% with "tone" (93.0% with "sentiment"). The
label changes for 3 sentences: 1 fixed and 2 broken. The p-value is 1.00.

In [ ]:
# Your code here.

**Answer here.** (1) Does this one change make a real difference to the
accuracy? (2) Does B3 show that the wording of the instruction never matters?
What would you test next?

*Answer here.*

## B4 — One limitation

**Answer here.** Your temperature `best_T` was chosen on movie-review
sentences. Would you use the same temperature to correct the model's
confidence on tweets about airlines? Explain why, and say what you would do
instead.

*Answer here.*

---
**Before you submit.** Restart the kernel and run every cell from top to
bottom. The scoring cells are fast the second time, because the scores are
saved.